# 01 — Exploratory Bias Analysis

Visualise IFS and AIFS 2m temperature forecast bias: distributions, seasonal patterns, and geographic structure.

**Data access:** All queries run through `PipelineDB` (DuckDB) — no full tables loaded into memory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..'))  # repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from nwp_census_eval.db import PipelineDB

# Override if data is synced locally:
# db = PipelineDB(aggregated_dir='/local/path/to/aggregated')
db = PipelineDB()
print('Registered views:', db.registered_views())

## Summary statistics by lead time

In [ ]:
ifs_stats = db.summary_stats(model='ifs')
aifs_stats = db.summary_stats(model='aifs')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, df) in zip(axes, [('IFS', ifs_stats), ('AIFS', aifs_stats)]):
    grp = df.groupby('lead_time')[['mean_bias', 'rmse']].mean()
    grp.plot(ax=ax, marker='o', markersize=4)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_title(f'{label} — mean bias & RMSE by lead time')
    ax.set_xlabel('Lead time (h)')
    ax.set_ylabel('K')
plt.tight_layout()

## Seasonal bias pattern (monthly × lead time)

In [ ]:
df_seasonal = db.query("""
    SELECT
        MONTH(valid_time) AS month,
        lead_time,
        AVG(bias)          AS mean_bias
    FROM ifs_bias
    GROUP BY MONTH(valid_time), lead_time
    ORDER BY month, lead_time
""")

pivot = df_seasonal.pivot(index='lead_time', columns='month', values='mean_bias')
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{lt}h' for lt in pivot.index])
ax.set_title('IFS mean bias — month × lead time')
plt.colorbar(im, ax=ax, label='Bias (K)')
plt.tight_layout()

## Geographic distribution of mean bias

In [ ]:
# County-level mean bias for 24h lead
df_county = db.query("""
    SELECT geo_id, AVG(bias) AS mean_bias
    FROM ifs_bias
    WHERE lead_time = 24
    GROUP BY geo_id
""")
print(f'{len(df_county):,} counties | bias range: {df_county.mean_bias.min():.2f} to {df_county.mean_bias.max():.2f} K')
df_county['mean_bias'].hist(bins=60, figsize=(8, 4))
plt.axvline(0, color='r', lw=1, ls='--')
plt.xlabel('Mean bias (K)')
plt.title('IFS 24h lead — county mean bias distribution')

## Interactive maps

Folium/Leaflet choropleths — hover any county for FIPS, name, and bias value.

In [ ]:
import sys
sys.path.insert(0, os.path.join("..", "scripts"))

import folium
import branca.colormap as cm
import geopandas as gpd
from IPython.display import display
from config import SHAPEFILE_PATH

# Load county geometries — keep GEOID, NAME, geometry only
gdf_counties = (
    gpd.read_file(SHAPEFILE_PATH)[["GEOID", "NAME", "geometry"]]
    .rename(columns={"GEOID": "geo_id"})
    .to_crs("EPSG:4326")
)
print(f"Loaded {len(gdf_counties):,} county geometries")


def bias_choropleth(df, title, vmin=-1.5, vmax=1.5):
    """Folium choropleth of mean_bias by county FIPS."""
    gdf = gdf_counties.merge(df[["geo_id", "mean_bias"]], on="geo_id", how="left")

    colormap = cm.LinearColormap(
        ["#2166ac", "#92c5de", "#f7f7f7", "#f4a582", "#d6604d"],
        vmin=vmin, vmax=vmax,
        caption=title,
    )

    m = folium.Map(location=[39, -96], zoom_start=4, tiles="CartoDB positron")

    def style_fn(feature):
        val = feature["properties"].get("mean_bias")
        return {
            "fillColor": colormap(val) if val is not None else "#d0d0d0",
            "color": "black",
            "weight": 0.2,
            "fillOpacity": 0.75,
        }

    tooltip = folium.GeoJsonTooltip(
        fields=["NAME", "geo_id", "mean_bias"],
        aliases=["County", "FIPS", "Mean bias (K)"],
        localize=True,
        sticky=False,
    )

    folium.GeoJson(gdf, style_function=style_fn, tooltip=tooltip).add_to(m)
    colormap.add_to(m)
    return m

### IFS mean bias by county

In [ ]:
# IFS bias at key lead times — 24h, 72h, 120h, 240h
for lt in [24, 72, 120, 240]:
    df_lt = db.query(f"""
        SELECT geo_id, AVG(bias) AS mean_bias
        FROM ifs_bias
        WHERE lead_time = {lt}
        GROUP BY geo_id
    """)
    df_lt["geo_id"] = df_lt["geo_id"].astype(str).str.zfill(5)
    print(f"IFS {lt}h — {len(df_lt):,} counties")
    display(bias_choropleth(df_lt, f"IFS {lt}h mean bias (K)"))

### IFS vs AIFS at 24h

In [ ]:
if "aifs_bias" in db.registered_views():
    df_ifs = db.query("""
        SELECT geo_id, AVG(bias) AS mean_bias
        FROM ifs_bias WHERE lead_time = 24
        GROUP BY geo_id
    """)
    df_aifs = db.query("""
        SELECT geo_id, AVG(bias) AS mean_bias
        FROM aifs_bias WHERE lead_time = 24
        GROUP BY geo_id
    """)
    df_ifs["geo_id"]  = df_ifs["geo_id"].astype(str).str.zfill(5)
    df_aifs["geo_id"] = df_aifs["geo_id"].astype(str).str.zfill(5)

    print("IFS 24h")
    display(bias_choropleth(df_ifs, "IFS 24h mean bias (K)"))
    print("AIFS 24h")
    display(bias_choropleth(df_aifs, "AIFS 24h mean bias (K)"))

    # Difference: AIFS bias − IFS bias
    df_diff = df_ifs.merge(df_aifs, on="geo_id", suffixes=("_ifs", "_aifs"))
    df_diff["mean_bias"] = df_diff["mean_bias_aifs"] - df_diff["mean_bias_ifs"]
    print("AIFS − IFS bias difference")
    display(bias_choropleth(df_diff, "AIFS − IFS bias (K)", vmin=-1.0, vmax=1.0))
else:
    print("aifs_bias view not registered — skipping IFS vs AIFS comparison")

### Helper: generic choropleth + area-weighted mean

`aland` (county land area m²) is embedded in every bias/anom table — all spatial means are area-weighted.

In [ ]:
def aw_mean(df, value_col):
    """Area-weighted spatial mean using the aland column."""
    w = df["aland"]
    return (df[value_col] * w).sum() / w.sum()


def map_choropleth(df, value_col, title, vmin=None, vmax=None, diverging=True):
    """Generic Folium choropleth. df must have geo_id and value_col columns."""
    df = df.copy()
    df["geo_id"] = df["geo_id"].astype(str).str.zfill(5)
    if vmin is None:
        vmin = df[value_col].quantile(0.02)
    if vmax is None:
        vmax = df[value_col].quantile(0.98)
    colors = (
        ["#2166ac", "#92c5de", "#f7f7f7", "#f4a582", "#d6604d"]
        if diverging
        else ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"]
    )
    gdf = gdf_counties.merge(df[["geo_id", value_col]], on="geo_id", how="left")
    colormap = cm.LinearColormap(colors, vmin=vmin, vmax=vmax, caption=title)
    m = folium.Map(location=[39, -96], zoom_start=4, tiles="CartoDB positron")

    def style_fn(feature):
        val = feature["properties"].get(value_col)
        return {
            "fillColor": colormap(val) if val is not None else "#d0d0d0",
            "color": "black",
            "weight": 0.2,
            "fillOpacity": 0.75,
        }

    tooltip = folium.GeoJsonTooltip(
        fields=["NAME", "geo_id", value_col],
        aliases=["County", "FIPS", title],
        localize=True,
        sticky=False,
    )
    folium.GeoJson(gdf, style_function=style_fn, tooltip=tooltip).add_to(m)
    colormap.add_to(m)
    return m

### Seasonal mean bias (IFS 24h)

In [ ]:
df_seas = db.query("""
    SELECT
        geo_id,
        CASE
            WHEN MONTH(valid_time) IN (12, 1, 2) THEN 'DJF'
            WHEN MONTH(valid_time) IN (3,  4, 5) THEN 'MAM'
            WHEN MONTH(valid_time) IN (6,  7, 8) THEN 'JJA'
            ELSE                                      'SON'
        END AS season,
        AVG(bias)      AS bias,
        AVG(abs_error) AS abs_error,
        AVG(aland)     AS aland
    FROM ifs_bias
    WHERE lead_time = 24
    GROUP BY geo_id, season
""")
df_seas["geo_id"] = df_seas["geo_id"].astype(str).str.zfill(5)

for season in ["DJF", "MAM", "JJA", "SON"]:
    d = df_seas[df_seas["season"] == season]
    print(f'{season} — area-weighted mean bias: {aw_mean(d, "bias"):+.3f} K')
    display(map_choropleth(d, "bias", f"IFS 24h bias — {season} (K)", vmin=-2, vmax=2))

### Monthly area-weighted mean bias timeseries (IFS 24h)

In [ ]:
df_mo = db.query("""
    SELECT
        MONTH(valid_time) AS month,
        geo_id,
        AVG(bias)  AS bias,
        AVG(aland) AS aland
    FROM ifs_bias
    WHERE lead_time = 24
    GROUP BY MONTH(valid_time), geo_id
""")
df_mo["geo_id"] = df_mo["geo_id"].astype(str).str.zfill(5)

mo_awm = (
    df_mo.groupby("month")
    .apply(lambda g: (g["bias"] * g["aland"]).sum() / g["aland"].sum())
    .rename("aw_mean_bias")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    mo_awm["month"], mo_awm["aw_mean_bias"],
    color=["#d6604d" if v > 0 else "#2166ac" for v in mo_awm["aw_mean_bias"]],
)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_xlabel("Month")
ax.set_ylabel("Area-weighted mean bias (K)")
ax.set_title("IFS 24h — area-weighted mean bias by month")
plt.tight_layout()

### Monthly bias maps — Jan, Apr, Jul, Oct (IFS 24h)

In [ ]:
for mo, label in [(1, "January"), (4, "April"), (7, "July"), (10, "October")]:
    d = df_mo[df_mo["month"] == mo]
    print(f'{label} — area-weighted mean bias: {aw_mean(d, "bias"):+.3f} K')
    display(map_choropleth(d, "bias", f"IFS 24h bias — {label} (K)", vmin=-2, vmax=2))

### Mean absolute error maps (IFS)

In [ ]:
for lt in [24, 120, 240]:
    df_mae = db.query(f"""
        SELECT geo_id, AVG(abs_error) AS abs_error, AVG(aland) AS aland
        FROM ifs_bias
        WHERE lead_time = {lt}
        GROUP BY geo_id
    """)
    df_mae["geo_id"] = df_mae["geo_id"].astype(str).str.zfill(5)
    print(f'IFS {lt}h — area-weighted MAE: {aw_mean(df_mae, "abs_error"):.3f} K')
    display(map_choropleth(df_mae, "abs_error", f"IFS {lt}h MAE (K)", vmin=0, vmax=4, diverging=False))

### Anomaly Correlation Coefficient (ACC)

Correlation between forecast and analysis anomalies (`fc_anom`, `an_anom`) from `ifs_anom`; computed per county across all available dates at each lead time.

In [ ]:
if "ifs_anom" in db.registered_views():
    for lt in [24, 120, 240]:
        df_acc = db.query(f"""
            SELECT
                geo_id,
                CORR(fc_anom, an_anom) AS acc,
                AVG(aland)             AS aland
            FROM ifs_anom
            WHERE lead_time = {lt}
            GROUP BY geo_id
        """)
        df_acc["geo_id"] = df_acc["geo_id"].astype(str).str.zfill(5)
        print(f'IFS {lt}h — area-weighted mean ACC: {aw_mean(df_acc, "acc"):.3f}')
        display(map_choropleth(df_acc, "acc", f"IFS {lt}h ACC", vmin=0, vmax=1, diverging=False))
else:
    print("ifs_anom view not registered — skipping ACC maps")